In [4]:
pip install tensorflow gradio numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\kanjut\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
import tensorflow as tf
import gradio as gr
import numpy as np

# LOAD EXISTING TRAINED MODEL

model = tf.keras.models.load_model("waste_model.keras")

# Waste Classification Dataset labels
CLASS_NAMES = ["Organic", "Recyclable"]

IMG_SIZE = 224

# PREDICTION FUNCTION

def predict_waste(image):

    if image is None:
        return "Please upload an image.", "", {}, ""

    # Convert image to RGB
    image = image.convert("RGB")

    # Resize image
    image = image.resize((IMG_SIZE, IMG_SIZE))

    # Convert image to numpy array
    image_array = np.array(image, dtype=np.float32)

    # Add batch dimension
    image_array = np.expand_dims(image_array, axis=0)

    # Predict using existing model
    predictions = model.predict(
        image_array,
        verbose=0
    )[0]

    # Get predicted class
    predicted_index = np.argmax(predictions)

    predicted_class = CLASS_NAMES[predicted_index]

    # Confidence score
    confidence = float(predictions[predicted_index] * 100)

    # Classification result
    if predicted_class == "Organic":

        result = "🟢 ORGANIC WASTE"

        recommendation = (
            "This waste is organic. "
            "It may be suitable for composting."
        )

    else:

        result = "♻️ RECYCLABLE WASTE"

        recommendation = (
            "This waste is recyclable. "
            "Check local recycling guidelines."
        )

    # Probability distribution
    probabilities = {
        CLASS_NAMES[i]: float(predictions[i])
        for i in range(len(CLASS_NAMES))
    }

    return (
        result,
        f"{confidence:.2f}%",
        probabilities,
        recommendation
    )

# GRADIO USER INTERFACE

with gr.Blocks(title="AI Waste Classification") as demo:

    gr.Markdown(
        """
        # ♻️ AI Waste Classification System

        Upload an image to classify waste using
        your already-trained TensorFlow model.

        **Categories:**
        - 🟢 Organic Waste
        - ♻️ Recyclable Waste
        """
    )

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                label="Upload Waste Image",
                type="pil",
                height=300
            )

            predict_button = gr.Button(
                "Classify Waste",
                variant="primary"
            )

            clear_button = gr.Button("Clear")

        with gr.Column():

            result_output = gr.Textbox(
                label="Classification Result"
            )

            confidence_output = gr.Textbox(
                label="Confidence Score"
            )

            probability_output = gr.Label(
                label="Prediction Probabilities",
                num_top_classes=2
            )

            recommendation_output = gr.Textbox(
                label="Disposal Recommendation"
            )

    # Prediction button
    predict_button.click(
        fn=predict_waste,
        inputs=image_input,
        outputs=[
            result_output,
            confidence_output,
            probability_output,
            recommendation_output
        ]
    )

    # Clear button
    clear_button.click(
        fn=lambda: (None, "", "", None, ""),
        inputs=[],
        outputs=[
            image_input,
            result_output,
            confidence_output,
            probability_output,
            recommendation_output
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
